# IA Aplicada a Lotofacil: Analise Estatistica e Previsao com Machine Learning

Este notebook demonstra como aplicar tecnicas de Inteligencia Artificial e analise estatistica ao historico real de sorteios da Lotofacil.

Utilizamos os seguintes conceitos:

- **Analise de frequencia** — quais numeros aparecem mais ao longo do historico
- **Analise de atraso** — ha quantos sorteios um numero nao aparece
- **Tendencia recente** — comportamento nos ultimos 30 sorteios
- **Score composto** — modelo ponderado para gerar uma sugestao de aposta
- **Arvore de Decisao** — classificacao supervisionada (quente x frio)

In [ ]:
# Instalar bibliotecas necessarias
!pip install scikit-learn pandas matplotlib seaborn --quiet

# Importar bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print("Bibliotecas importadas com sucesso!")

## Etapa 1 — Carregar os Dados Historicos

O arquivo CSV contem o historico dos sorteios da Lotofacil com os 15 numeros sorteados em cada concurso, ordenados de forma crescente.

Fazemos o upload do arquivo no Colab e carregamos usando o pandas, informando o separador correto (ponto e virgula).

In [ ]:
# Carregar o arquivo CSV com o historico de sorteios
# Faca o upload do arquivo loteria.csv no Colab antes de rodar esta celula

df = pd.read_csv('loteria.csv', sep=';')

# Verificar se o carregamento foi correto
# O shape deve ser (293, 17): 293 sorteios e 17 colunas (Concurso, Data + 15 bolas)
print(f"Shape do DataFrame: {df.shape}")

# Identificar as colunas que representam as bolas sorteadas
ball_cols = [c for c in df.columns if 'bola' in c]

print(f"Total de sorteios carregados: {len(df)}")
print(f"Concursos de {df['Concurso'].min()} a {df['Concurso'].max()}")
print(f"Colunas de bolas: {ball_cols}")
print()
df.head()

## Etapa 2 — Analise de Frequencia

A **frequencia** indica quantas vezes cada numero foi sorteado ao longo de todos os concursos.

Na Lotofacil sao sorteados 15 de 25 numeros, portanto a probabilidade teorica de cada numero aparecer em um sorteio e de **60%**. Numeros com frequencia muito acima ou abaixo desse valor podem indicar tendencias no historico.

In [ ]:
# Calcular a frequencia de cada numero (1 a 25)

all_numbers = []
for _, row in df.iterrows():
    for c in ball_cols:
        all_numbers.append(int(row[c]))

freq = Counter(all_numbers)

freq_df = pd.DataFrame({
    'numero':     list(range(1, 26)),
    'frequencia': [freq[n] for n in range(1, 26)],
    'percentual': [round(freq[n] / len(df) * 100, 1) for n in range(1, 26)]
})

print("Frequencia de cada numero:")
print(freq_df.to_string(index=False))

In [ ]:
# Visualizar a frequencia como grafico de barras

plt.figure(figsize=(14, 5))
cores = ['#2196F3' if freq[n] >= freq_df['frequencia'].median() else '#90CAF9'
         for n in range(1, 26)]
bars = plt.bar([str(n).zfill(2) for n in range(1, 26)],
               [freq[n] for n in range(1, 26)],
               color=cores, edgecolor='white', linewidth=0.5)

# Linha de media
media = freq_df['frequencia'].mean()
plt.axhline(media, color='red', linestyle='--', linewidth=1.5, label=f'Media = {media:.0f}')

plt.title('Frequencia Historica de Cada Numero — Lotofacil', fontsize=14, pad=15)
plt.xlabel('Numero')
plt.ylabel('Quantidade de sorteios')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Numero mais sorteado: {freq_df.loc[freq_df['frequencia'].idxmax(), 'numero']} "
      f"({freq_df['frequencia'].max()} vezes)")
print(f"Numero menos sorteado: {freq_df.loc[freq_df['frequencia'].idxmin(), 'numero']} "
      f"({freq_df['frequencia'].min()} vezes)")

## Etapa 3 — Analise de Atraso

O **atraso** indica ha quantos sorteios consecutivos um numero não aparece, contando a partir do concurso mais recente.

Como o DataFrame esta ordenado do sorteio mais recente (indice 0) para o mais antigo, o proprio indice da primeira ocorrencia encontrada ja representa o atraso em numero de sorteios.

In [ ]:
# Calcular o atraso de cada numero
# O DataFrame esta em ordem decrescente: indice 0 = sorteio mais recente
# Portanto o indice da primeira aparicao encontrada equivale ao atraso em sorteios

delay = {}
for n in range(1, 26):
    found = False
    for idx, row in df.iterrows():
        nums = set(int(row[c]) for c in ball_cols)
        if n in nums:
            delay[n] = idx  # indice == quantidade de sorteios sem aparecer
            found = True
            break
    if not found:
        delay[n] = len(df)

delay_df = pd.DataFrame({
    'numero': list(range(1, 26)),
    'atraso': [delay[n] for n in range(1, 26)]
})

print("Atraso de cada numero (em sorteios):")
print(delay_df.to_string(index=False))
print(f"\nMaior atraso: numero {delay_df.loc[delay_df['atraso'].idxmax(), 'numero']} "
      f"({delay_df['atraso'].max()} sorteios)")

In [ ]:
# Visualizar o atraso como grafico de barras com cores por categoria

plt.figure(figsize=(14, 5))
cores_atraso = ['#F44336' if delay[n] >= 2 else '#FF9800' if delay[n] == 1 else '#4CAF50'
                for n in range(1, 26)]
plt.bar([str(n).zfill(2) for n in range(1, 26)],
        [delay[n] for n in range(1, 26)],
        color=cores_atraso, edgecolor='white')

p1 = mpatches.Patch(color='#4CAF50', label='Atraso 0 (apareceu no ultimo sorteio)')
p2 = mpatches.Patch(color='#FF9800', label='Atraso 1')
p3 = mpatches.Patch(color='#F44336', label='Atraso 2 ou mais')
plt.legend(handles=[p1, p2, p3], fontsize=9)

plt.title('Atraso de Cada Numero — Sorteios Consecutivos sem Aparecer', fontsize=13, pad=15)
plt.xlabel('Numero')
plt.ylabel('Atraso (sorteios)')
plt.tight_layout()
plt.show()

## Etapa 4 — Tendencia Recente (Ultimos 30 Sorteios)

A tendencia recente captura o comportamento de **curto prazo** dos numeros, analisando apenas os ultimos 30 concursos.

Esse criterio serve para identificar numeros que estao em um momento de alta frequencia, independentemente do historico total.

In [ ]:
# Calcular frequencia nos ultimos 30 sorteios

recent = df.head(30)  # os 30 mais recentes estao no inicio do DataFrame
recent_counts = Counter()
for _, row in recent.iterrows():
    for c in ball_cols:
        recent_counts[int(row[c])] += 1

rec_df = pd.DataFrame({
    'numero':      list(range(1, 26)),
    'freq_recente': [recent_counts.get(n, 0) for n in range(1, 26)]
})

# Heatmap em formato de grade 5x5 (como um volante)
matriz = np.array([recent_counts.get(n, 0) for n in range(1, 26)]).reshape(5, 5)
labels_mat = np.array([str(n).zfill(2) for n in range(1, 26)]).reshape(5, 5)

plt.figure(figsize=(8, 6))
ax = sns.heatmap(matriz, annot=labels_mat, fmt='', cmap='YlOrRd',
                 linewidths=2, linecolor='white',
                 cbar_kws={'label': 'Aparicoes nos ultimos 30 sorteios'},
                 annot_kws={'size': 14, 'weight': 'bold'})

# Adicionar contagem abaixo do numero em cada celula
for i in range(5):
    for j in range(5):
        n = i * 5 + j + 1
        ax.text(j + 0.5, i + 0.75, f'({recent_counts.get(n, 0)}x)',
                ha='center', va='center', fontsize=9, color='#555')

plt.title('Mapa de Calor — Frequencia nos Ultimos 30 Sorteios', fontsize=13, pad=15)
plt.axis('off')
plt.tight_layout()
plt.show()

## Etapa 5 — Score Composto (Modelo Ponderado)

Combinamos os tres criterios em um **score ponderado** para ranquear os 25 numeros:

| Criterio | Peso |
|---|---|
| Frequencia historica | 40% |
| Atraso (pressao estatistica) | 30% |
| Tendencia recente | 30% |

Cada criterio e **normalizado** para um valor entre 0 e 1 antes de ser multiplicado pelo seu peso. Isso garante que nenhum criterio domine os outros por diferenca de escala.

In [ ]:
# Calcular o score composto para cada numero

max_freq  = max(freq.values())
max_delay = max(delay.values())
max_rec   = max(recent_counts.values())

scores = {}
for n in range(1, 26):
    norm_freq  = freq[n] / max_freq
    norm_delay = delay[n] / max_delay
    norm_rec   = recent_counts.get(n, 0) / max_rec
    scores[n]  = 0.40 * norm_freq + 0.30 * norm_delay + 0.30 * norm_rec

score_df = pd.DataFrame({
    'numero':      list(range(1, 26)),
    'freq_hist':   [freq[n] for n in range(1, 26)],
    'atraso':      [delay[n] for n in range(1, 26)],
    'freq_rec30':  [recent_counts.get(n, 0) for n in range(1, 26)],
    'score_final': [round(scores[n], 4) for n in range(1, 26)]
}).sort_values('score_final', ascending=False)

print("Ranking por Score Composto:")
print(score_df.to_string(index=False))

In [ ]:
# Selecionar os 15 numeros com maior score

top15 = sorted(score_df.head(15)['numero'].tolist())
print("Combinacao sugerida (15 numeros com maior score):")
print()
print("  " + "  ".join([str(n).zfill(2) for n in top15]))
print()

# Visualizar o ranking como grafico de barras
plt.figure(figsize=(14, 5))
nums_ord   = score_df['numero'].tolist()
scores_ord = score_df['score_final'].tolist()
cores_sc   = ['#1976D2' if n in top15 else '#B0BEC5' for n in nums_ord]

plt.bar([str(n).zfill(2) for n in nums_ord], scores_ord, color=cores_sc, edgecolor='white')
plt.axhline(scores_ord[14], color='red', linestyle='--', linewidth=1.5,
            label=f'Corte top-15 (score = {scores_ord[14]:.4f})')

p1 = mpatches.Patch(color='#1976D2', label='Top 15 selecionados')
p2 = mpatches.Patch(color='#B0BEC5', label='Nao selecionados')
plt.legend(handles=[p1, p2])

plt.title('Score Composto por Numero — Modelo Ponderado', fontsize=13, pad=15)
plt.xlabel('Numero (ordenado por score)')
plt.ylabel('Score final (0 a 1)')
plt.tight_layout()
plt.show()

## Etapa 6 — Arvore de Decisao: Classificar Numeros como Quente ou Frio

Agora aplicamos **Machine Learning supervisionado**: treinamos uma **Arvore de Decisao** para classificar cada numero como:

- **Quente** — score maior ou igual a mediana (alta chance de selecao)
- **Frio** — score abaixo da mediana (menor chance de selecao)

Os **atributos (features)** do modelo sao: frequencia historica, atraso e frequencia recente. O rotulo (classe) e definido com base no score composto calculado na etapa anterior.

In [ ]:
# Preparar o dataset para treinar a Arvore de Decisao

ml_df = pd.DataFrame({
    'numero':     list(range(1, 26)),
    'freq_hist':  [freq[n] for n in range(1, 26)],
    'atraso':     [delay[n] for n in range(1, 26)],
    'freq_rec30': [recent_counts.get(n, 0) for n in range(1, 26)],
    'score':      [scores[n] for n in range(1, 26)]
})

# Rotulo: 'quente' se score >= mediana, 'frio' caso contrario
mediana_score = ml_df['score'].median()
ml_df['classe'] = ml_df['score'].apply(lambda x: 'quente' if x >= mediana_score else 'frio')

print("Dataset para a Arvore de Decisao:")
print(ml_df.to_string(index=False))
print(f"\nMediana do score = {mediana_score:.4f}")
print(f"Numeros 'quentes': {ml_df[ml_df['classe']=='quente']['numero'].tolist()}")
print(f"Numeros 'frios':   {ml_df[ml_df['classe']=='frio']['numero'].tolist()}")

In [ ]:
# Separar atributos (X) e rotulo (y)

X = ml_df[['freq_hist', 'atraso', 'freq_rec30']]  # atributos
y = ml_df['classe']                                 # rotulo

# Dividir em treino (70%) e teste (30%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Treino: {len(X_train)} exemplos | Teste: {len(X_test)} exemplos")

In [ ]:
# Criar e treinar o modelo de Arvore de Decisao

modelo = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo.fit(X_train, y_train)

print("Modelo treinado com sucesso!")

In [ ]:
# Avaliar o modelo no conjunto de teste

y_pred = modelo.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Acuracia do modelo: {acc:.2%}")
print()
print("Relatorio de Classificacao:")
print(classification_report(y_test, y_pred))

In [ ]:
# Visualizar a Arvore de Decisao

plt.figure(figsize=(14, 7))
plot_tree(
    modelo,
    feature_names=['freq_hist', 'atraso', 'freq_rec30'],
    class_names=modelo.classes_,
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title('Arvore de Decisao — Classificacao de Numeros Quentes e Frios', fontsize=13, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Classificar todos os 25 numeros com o modelo treinado

ml_df['predicao'] = modelo.predict(X)
quentes_ml = sorted(ml_df[ml_df['predicao'] == 'quente']['numero'].tolist())

print("Classificacao de todos os numeros pela Arvore de Decisao:")
print()
print(ml_df[['numero', 'freq_hist', 'atraso', 'freq_rec30', 'score', 'classe', 'predicao']].to_string(index=False))
print()
print(f"Numeros classificados como 'quente' pelo modelo: {quentes_ml}")

In [ ]:
# Fazer uma previsao individual para um numero especifico
# Exemplo: classificar o numero 7

numero_exemplo = 7
dados_exemplo = pd.DataFrame({
    'freq_hist':  [freq[numero_exemplo]],
    'atraso':     [delay[numero_exemplo]],
    'freq_rec30': [recent_counts.get(numero_exemplo, 0)]
})

resultado = modelo.predict(dados_exemplo)[0]
print(f"Numero analisado: {numero_exemplo}")
print(f"  Frequencia historica : {freq[numero_exemplo]}")
print(f"  Atraso               : {delay[numero_exemplo]} sorteios")
print(f"  Frequencia rec. (30) : {recent_counts.get(numero_exemplo, 0)}")
print(f"  Classificacao        : {resultado}")

## Etapa 7 — Combinacao Final Sugerida

Combinamos o **score composto** com a **classificacao da Arvore de Decisao** para apresentar a sugestao final de 15 numeros para o proximo sorteio.

In [ ]:
# Exibir a combinacao final sugerida

top15 = sorted(score_df.head(15)['numero'].tolist())

print("=" * 50)
print("COMBINACAO SUGERIDA PARA O PROXIMO SORTEIO")
print("=" * 50)
print()
print("  " + "  ".join([str(n).zfill(2) for n in top15]))
print()
print("Metodologia utilizada:")
print("  - Frequencia historica (293 sorteios) com peso de 40%")
print("  - Atraso (pressao estatistica)        com peso de 30%")
print("  - Tendencia recente (30 sorteios)      com peso de 30%")
print()

In [ ]:
# Visualizar a combinacao final como volante grafico

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 5)
ax.set_ylim(0, 5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_facecolor('#f5f5f5')
fig.patch.set_facecolor('#f5f5f5')

for i, n in enumerate(range(1, 26)):
    row = i // 5
    col = i % 5
    x = col + 0.5
    y = 4.5 - row
    selecionado = n in top15
    cor = '#1565C0' if selecionado else '#E0E0E0'
    cor_texto = 'white' if selecionado else '#757575'
    circle = plt.Circle((x, y), 0.38, color=cor, zorder=2)
    ax.add_patch(circle)
    ax.text(x, y, str(n).zfill(2), ha='center', va='center',
            fontsize=14, fontweight='bold', color=cor_texto, zorder=3)

ax.set_title('Volante — Numeros Sugeridos pelo Modelo de IA',
             fontsize=13, pad=20, fontweight='bold')

p1 = mpatches.Patch(color='#1565C0', label='Selecionado')
p2 = mpatches.Patch(color='#E0E0E0', label='Nao selecionado')
ax.legend(handles=[p1, p2], loc='lower center',
          bbox_to_anchor=(0.5, -0.05), ncol=2, frameon=False)

plt.tight_layout()
plt.show()